# Two-stage pipeline — is there a technically valid way for Model 1 (mileage) to help Model 2 (defects)?  (experiments only)

**Why this matters.** The SRS (FR-3) asks Stage 1 to estimate kilometres and Stage 2 to use that estimate. Notebook 04 found that, with the UK data, an estimated odometer adds nothing to a model that already has age and fuel
(the true odometer adds about +0.015 to +0.018 AUC). This notebook asks, with data and without forcing anything, **in which situations a Stage-1 output carries information that Stage 2 does not already have.**

**A reason to expect "nothing", and where it can fail.** If Stage 1 is a function of the same registry fields that Stage 2 receives, Stage 2 already has everything Stage 1 knows (a tree can re-create any such function). Stage 1 can only add information if
1. it is trained on **information Stage 2's training data lacks** (a larger or different odometer data set: "external knowledge"), which matters most when Stage 2 has **few labels**, or for **simple models** (e.g. Logistic Regression) that cannot build the interaction themselves; or
2. its output is **replaced by a measured odometer** whenever one exists (maintenance records, user input), so that the pipeline uses the real value if available and the estimate otherwise.

Each is tested below. Where no gain is found, it is reported as no gain.

**Method corrections in this notebook (relative to notebook 04).**
* Stage-1 estimates fed to Stage 2 for the *training* rows are **out-of-fold** (5-fold by vehicle). In notebook 04 they were in-sample, which makes the training feature look more accurate than it will be at inference.
* Age is the **integer** year difference (the registry has the year of manufacture only), as in notebooks 05/06.
* Differences are given with a paired bootstrap interval over vehicles: a gain smaller than its interval is not reported as a gain.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.options.display.width = 190; pd.options.display.max_columns = 40; pd.options.display.float_format = "{:,.4f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from real_outcome_data import load_both
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from xgboost import XGBClassifier, XGBRegressor
try:
    display
except NameError:
    display = print
uk_all, nl_all = load_both()
d = uk_all[uk_all.age_year.between(3, 25) & uk_all.fuel.isin(["petrol", "diesel", "hybrid", "electric"])].copy()
d["make_c"] = d.make.where(d.make.isin(d.make.value_counts().head(60).index), "OTHER")
d["model_c"] = d.model.where(d.model.isin(d.model.value_counts().head(400).index), "OTHER")
CATS = {c: sorted(d[c].unique()) for c in ["fuel", "make_c", "model_c"]}
def prep(df, cols):
    X = df[cols].copy()
    for c in cols:
        if c in CATS: X[c] = pd.Categorical(X[c], categories=CATS[c])
    return X
TG = {"brakes": "brakes", "electrical": "electrical", "engine-related": "engine_emissions"}
S1F = ["age_year", "fuel", "make_c"]                    # Stage-1 inputs = what the Israeli registry gives for any plate
S2G = ["age_year", "fuel", "make_c", "model_c"]          # Stage-2 inputs without mileage
print(f"UK table: {len(d):,} tests, {d.unit.nunique():,} vehicles; odometer median {d.km.median():,.0f} km")

UK table: 367,911 tests, 367,738 vehicles; odometer median 101,426 km


## 1. How much can Stage 1 know? Quality of the mileage estimate by input set (real odometers, held-out vehicles)

In [2]:
a_i, t_i = next(GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=0).split(d, groups=d.unit))
A, T = d.iloc[a_i].copy(), d.iloc[t_i].copy()
def s1_model(): return XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08, subsample=0.8, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0)
rows = []
for nm, cols in [("age only", ["age_year"]), ("age, fuel", ["age_year", "fuel"]), ("age, fuel, make  (Israeli registry)", S1F),
                 ("age, fuel, make, model  (Israeli registry)", S1F + ["model_c"]), ("+ engine size (NOT in the Israeli registry resource)", S1F + ["model_c", "cc"])]:
    m = s1_model().fit(prep(A, cols), A.km); pt = m.predict(prep(T, cols))
    rows.append({"Stage-1 inputs": nm, "R2 (held-out vehicles)": r2_score(T.km, pt), "RMSE km": float(np.sqrt(np.mean((T.km - pt) ** 2))), "MAE km": float(np.mean(np.abs(T.km - pt)))})
display(pd.DataFrame(rows))
print("Stage-1 targets are real odometer readings of ~370k cars (UK, miles converted to km).")

,Stage-1 inputs,R2 (held-out vehicles),RMSE km,MAE km
0,age only,0.3578,"58,821.8937","42,049.4680"
1,"age, fuel",0.4743,"53,219.9165","37,872.4821"
2,"age, fuel, make (Israeli registry)",0.5038,"51,703.6997","36,803.8361"
3,"age, fuel, make, model (Israeli registry)",0.5463,"49,442.9908","35,206.1800"
4,+ engine size (NOT in the Israeli registry res...,0.5494,"49,269.1801","35,085.6357"


Stage-1 targets are real odometer readings of ~370k cars (UK, miles converted to km).


## 2. Main pipeline test: no mileage vs estimated mileage (SRS pipeline) vs true mileage
`E (out-of-fold)` is the SRS pipeline done correctly. `E (in-sample)` reproduces what notebook 04 did, to show whether that mattered.

In [3]:
def fit_cls(X, y, depth=6, n=250, mcw=20):
    return XGBClassifier(n_estimators=n, max_depth=depth, learning_rate=0.06, subsample=0.8, colsample_bytree=0.8, min_child_weight=mcw, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0).fit(X, y)
def ece(y, p, bins=10):
    q = pd.qcut(p, bins, duplicates="drop"); t = pd.DataFrame({"y": y, "p": p}).groupby(q, observed=True).agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return float((abs(t.p - t.y) * t.n).sum() / t.n.sum())
def paired_boot(y, p0, p1, units, n=100, seed=0):
    rng = np.random.default_rng(seed); u = pd.Series(np.arange(len(y))).groupby(units.to_numpy()).apply(lambda s: s.to_numpy()); keys = u.index.to_numpy(); out = []
    for _ in range(n):
        ii = np.concatenate(u.loc[rng.choice(keys, len(keys))].to_list()); out.append(roc_auc_score(y[ii], p1[ii]) - roc_auc_score(y[ii], p0[ii]))
    return np.percentile(out, [2.5, 97.5])

# Stage 1 on A -> predictions for T; out-of-fold predictions for A (5 folds by vehicle)
s1 = s1_model().fit(prep(A, S1F), A.km)
T["km_est"] = s1.predict(prep(T, S1F)); A["km_in"] = s1.predict(prep(A, S1F))
A["km_oof"] = np.nan
for tr_i, va_i in GroupKFold(n_splits=5).split(A, groups=A.unit):
    A.iloc[va_i, A.columns.get_loc("km_oof")] = s1_model().fit(prep(A.iloc[tr_i], S1F), A.iloc[tr_i].km).predict(prep(A.iloc[va_i], S1F))
print(f"Stage-1 R2 on test vehicles: {r2_score(T.km, T.km_est):.3f} | out-of-fold on training vehicles: {r2_score(A.km, A.km_oof):.3f} | in-sample on training vehicles: {r2_score(A.km, A.km_in):.3f}")
rows, P = [], {}
for tname, tcol in TG.items():
    y = T[tcol].to_numpy()
    sc = {"no mileage (age, fuel, make, model)": (S2G, A, T),
          "E out-of-fold (SRS pipeline)": (S2G + ["km_est"], A.assign(km_est=A.km_oof), T),
          "E in-sample (as in notebook 04)": (S2G + ["km_est"], A.assign(km_est=A.km_in), T),
          "M true odometer": (S2G + ["km"], A, T)}
    for nm, (cols, tr, te) in sc.items():
        m = fit_cls(prep(tr, cols), tr[tcol]); p = m.predict_proba(prep(te, cols))[:, 1]; P[(tname, nm)] = p
        rows.append({"system": tname, "scenario": nm, "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "ECE": ece(y, p)})
    # M-trained model fed with the ESTIMATE at inference (what happens if the real odometer is missing and Stage 1 fills in)
    cols = S2G + ["km"]; m = fit_cls(prep(A, cols), A[tcol]); p = m.predict_proba(prep(T.assign(km=T.km_est), cols))[:, 1]; P[(tname, "M-trained, E at inference")] = p
    rows.append({"system": tname, "scenario": "M-trained model, estimate at inference", "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p), "ECE": ece(y, p)})
R2 = pd.DataFrame(rows); display(R2)
rows = []
for tname, tcol in TG.items():
    y = T[tcol].to_numpy(); base = P[(tname, "no mileage (age, fuel, make, model)")]
    for nm in ["E out-of-fold (SRS pipeline)", "M true odometer", "M-trained, E at inference"]:
        lo, hi = paired_boot(y, base, P[(tname, nm)], T.unit)
        rows.append({"system": tname, "scenario vs 'no mileage'": nm, "AUC difference": roc_auc_score(y, P[(tname, nm)]) - roc_auc_score(y, base), "95% CI low": lo, "95% CI high": hi})
print("Paired bootstrap over vehicles of the AUC difference against the model that has no mileage at all:")
display(pd.DataFrame(rows))

Stage-1 R2 on test vehicles: 0.504 | out-of-fold on training vehicles: 0.506 | in-sample on training vehicles: 0.514


,system,scenario,ROC-AUC,PR-AUC,ECE
0,brakes,"no mileage (age, fuel, make, model)",0.6912,0.1284,0.0037
1,brakes,E out-of-fold (SRS pipeline),0.6902,0.1282,0.0050
2,brakes,E in-sample (as in notebook 04),0.6898,0.1282,0.0048
3,brakes,M true odometer,0.7056,0.1355,0.0047
4,brakes,"M-trained model, estimate at inference",0.6880,0.1271,0.0069
5,electrical,"no mileage (age, fuel, make, model)",0.7259,0.2197,0.0037
6,electrical,E out-of-fold (SRS pipeline),0.7256,0.2190,0.0040
7,electrical,E in-sample (as in notebook 04),0.7257,0.2189,0.0042
8,electrical,M true odometer,0.7423,0.2351,0.0037
9,electrical,"M-trained model, estimate at inference",0.7233,0.2171,0.0081


Paired bootstrap over vehicles of the AUC difference against the model that has no mileage at all:


,system,scenario vs 'no mileage',AUC difference,95% CI low,95% CI high
0,brakes,E out-of-fold (SRS pipeline),-0.0011,-0.0019,-0.0003
1,brakes,M true odometer,0.0144,0.0114,0.0171
2,brakes,"M-trained, E at inference",-0.0033,-0.0046,-0.0020
3,electrical,E out-of-fold (SRS pipeline),-0.0003,-0.0008,0.0002
4,electrical,M true odometer,0.0164,0.0141,0.0184
5,electrical,"M-trained, E at inference",-0.0026,-0.0038,-0.0018
6,engine-related,E out-of-fold (SRS pipeline),0.0006,-0.0004,0.0018
7,engine-related,M true odometer,0.0161,0.0125,0.0201
8,engine-related,"M-trained, E at inference",-0.0088,-0.0111,-0.0065


## 3. Valid design 1 — "odometer if known, estimate otherwise" (Stage 1 as an imputer)
Stage 2 is trained on a mix of true and estimated odometers **with a flag saying which one it got** (the backend knows whether the value came from a maintenance record / the mechanic or from Stage 1).
At inference a fraction `f` of the vehicles has a real odometer; the others get the Stage-1 estimate. Comparison: a model trained only on true odometers and fed estimates when they are missing.

In [4]:
rng = np.random.default_rng(0)
mix = rng.random(len(A)) < 0.5                                 # training: half of the rows use the (out-of-fold) estimate, flagged
A["km_mixed"] = np.where(mix, A.km_oof, A.km); A["km_is_real"] = (~mix).astype(int)
fr = {}
rows = []
for tname, tcol in TG.items():
    y = T[tcol].to_numpy()
    mA = fit_cls(prep(A, S2G + ["km"]), A[tcol])                                     # trained on true odometers only
    mB = fit_cls(prep(A.assign(km=A.km_mixed), S2G + ["km", "km_is_real"]), A[tcol])  # trained on mixed + flag
    g = fit_cls(prep(A, S2G), A[tcol]); pg = g.predict_proba(prep(T, S2G))[:, 1]
    for f in [0, 0.25, 0.5, 0.75, 1.0]:
        known = np.random.default_rng(1).random(len(T)) < f
        Tm = T.assign(km=np.where(known, T.km, T.km_est), km_is_real=known.astype(int))
        pA = mA.predict_proba(prep(Tm, S2G + ["km"]))[:, 1]; pB = mB.predict_proba(prep(Tm, S2G + ["km", "km_is_real"]))[:, 1]
        rows.append({"system": tname, "share with real odometer": f, "no-mileage model": roc_auc_score(y, pg), "trained on true only": roc_auc_score(y, pA), "trained on mixed + flag": roc_auc_score(y, pB),
                     "ECE (true-only)": ece(y, pA), "ECE (mixed+flag)": ece(y, pB)})
display(pd.DataFrame(rows))

,system,share with real odometer,no-mileage model,trained on true only,trained on mixed + flag,ECE (true-only),ECE (mixed+flag)
0,brakes,0.0000,0.6912,0.6880,0.6903,0.0069,0.0040
1,brakes,0.2500,0.6912,0.6929,0.6944,0.0060,0.0035
2,brakes,0.5000,0.6912,0.6955,0.6971,0.0053,0.0041
3,brakes,0.7500,0.6912,0.7008,0.7015,0.0048,0.0039
4,brakes,1.0000,0.6912,0.7056,0.7056,0.0047,0.0039
5,electrical,0.0000,0.7259,0.7233,0.7251,0.0081,0.0032
6,electrical,0.2500,0.7259,0.7278,0.7290,0.0068,0.0040
7,electrical,0.5000,0.7259,0.7328,0.7329,0.0055,0.0037
8,electrical,0.7500,0.7259,0.7375,0.7371,0.0045,0.0044
9,electrical,1.0000,0.7259,0.7423,0.7411,0.0037,0.0037


## 4. Valid design 2 — mileage *relative to what Stage 1 expects* (usage intensity)
Physically meaningful: a car with far more kilometres than typical for its age is worn more. This uses Stage 1 as the **reference** and needs a real odometer. Question: does the explicit ratio add anything beyond the raw odometer?

In [5]:
A["usage"] = A.km / A.km_oof; T["usage"] = T.km / T.km_est
rows = []
for tname, tcol in TG.items():
    y = T[tcol].to_numpy()
    for nm, cols in [("no mileage", S2G), ("true odometer (km)", S2G + ["km"]), ("usage ratio only (km / Stage-1 expectation)", S2G + ["usage"]), ("km + usage ratio", S2G + ["km", "usage"])]:
        m = fit_cls(prep(A, cols), A[tcol]); p = m.predict_proba(prep(T, cols))[:, 1]; rows.append({"system": tname, "features": nm, "ROC-AUC": roc_auc_score(y, p), "PR-AUC": average_precision_score(y, p)})
display(pd.DataFrame(rows).pivot(index="features", columns="system", values="ROC-AUC")[list(TG)])

system,brakes,electrical,engine-related
features,,,
km + usage ratio,0.7059,0.7422,0.7497
no mileage,0.6912,0.7259,0.7329
true odometer (km),0.7056,0.7423,0.7490
usage ratio only (km / Stage-1 expectation),0.7052,0.7412,0.7490


## 5. How good would Stage 1 have to be? (simulation, not a model)
`km_lambda = lambda * true odometer + (1 - lambda) * Stage-1 estimate`, same for training (out-of-fold) and test rows. It shows how the Stage-2 AUC would grow if Stage 1 explained more of the mileage than the registry fields allow.
This is a sensitivity analysis: no real Stage 1 reaches the high-R2 rows with registry-only inputs.

In [6]:
rows = []
for lam in [0, 0.25, 0.5, 0.75, 1.0]:
    A["kl"] = lam * A.km + (1 - lam) * A.km_oof; T["kl"] = lam * T.km + (1 - lam) * T.km_est
    r2 = r2_score(T.km, T.kl)
    for tname, tcol in TG.items():
        y = T[tcol].to_numpy(); m = fit_cls(prep(A, S2G + ["kl"]), A[tcol]); rows.append({"lambda": lam, "R2 of the mileage input vs truth": r2, "system": tname, "ROC-AUC": roc_auc_score(y, m.predict_proba(prep(T, S2G + ["kl"]))[:, 1])})
sens = pd.DataFrame(rows).pivot_table(index=["lambda", "R2 of the mileage input vs truth"], columns="system", values="ROC-AUC")[list(TG)]; display(sens)

,system,brakes,electrical,engine-related
lambda,R2 of the mileage input vs truth,,,
0.0000,0.5038,0.6902,0.7256,0.7335
0.2500,0.7209,0.7023,0.7398,0.7486
0.5000,0.8760,0.7060,0.7418,0.7511
0.7500,0.9690,0.7058,0.7421,0.7508
1.0000,1.0000,0.7056,0.7423,0.7490


## 6. Valid design 3 — Stage 1 trained on *different* data than Stage 2 (external knowledge), small Stage-2 training sets
The realistic Israeli situation: Stage 1 is trained on odometer data (no defect labels); Stage 2 is trained on inspection labels, possibly few. Vehicles are split into three disjoint parts:
**S1** (40 %, Stage 1 sees only odometers) · **S2** (30 %, Stage-2 training pool, labels) · **T** (30 %, test). Stage-2 training sets of increasing size are drawn from S2 (3 random draws each; the same test part for all).
Both XGBoost and Logistic Regression (with a spline on age, to be fair to it) are tried, with and without the Stage-1 estimate. The test part and all hyper-parameters are fixed across variants, so each pair differs only by that one feature.

In [7]:
p1, p2 = next(GroupShuffleSplit(n_splits=1, train_size=0.4, random_state=5).split(d, groups=d.unit)); rest = d.iloc[p2]
q2, q3 = next(GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=5).split(rest, groups=rest.unit))
S1, S2, T3 = d.iloc[p1].copy(), rest.iloc[q2].copy(), rest.iloc[q3].copy()
assert not (set(S1.unit) & set(S2.unit)) and not (set(S1.unit) & set(T3.unit)) and not (set(S2.unit) & set(T3.unit))
s1b = s1_model().fit(prep(S1, S1F), S1.km)
for df in (S2, T3): df["km_est"] = s1b.predict(prep(df, S1F))
print(f"parts: S1 {len(S1):,} | S2 pool {len(S2):,} | T {len(T3):,} rows. Stage-1 R2 on T: {r2_score(T3.km, T3.km_est):.3f}")
def lr_pipe(with_e):
    num = [("age", SplineTransformer(n_knots=6, degree=3), ["age_year"])] + ([("e", StandardScaler(), ["km_est"])] if with_e else [])
    return Pipeline([("p", ColumnTransformer(num + [("c", OneHotEncoder(handle_unknown="ignore", min_frequency=30, sparse_output=False), ["fuel", "make_c"])])), ("m", LogisticRegression(max_iter=2000))])
def run_size(tr, te, tcol, with_e, kind):
    y = te[tcol].to_numpy()
    if kind == "XGBoost":
        cols = S2G + (["km_est"] if with_e else []); m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=20, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0).fit(prep(tr, cols), tr[tcol])
        return roc_auc_score(y, m.predict_proba(prep(te, cols))[:, 1])
    cols = ["age_year", "fuel", "make_c"] + (["km_est"] if with_e else []); m = lr_pipe(with_e).fit(tr[cols].astype({"fuel": str, "make_c": str}), tr[tcol])
    return roc_auc_score(y, m.predict_proba(te[cols].astype({"fuel": str, "make_c": str}))[:, 1])
rows = []
for n in [1000, 3000, 10000, 30000, len(S2)]:
    for rep in range(3 if n < len(S2) else 1):
        tr = S2.sample(n, random_state=rep) if n < len(S2) else S2
        for tname, tcol in TG.items():
            for kind in ["XGBoost", "Logistic Regression"]:
                a0, a1 = run_size(tr, T3, tcol, False, kind), run_size(tr, T3, tcol, True, kind)
                rows.append({"Stage-2 training size": n, "rep": rep, "system": tname, "model": kind, "AUC without Stage 1": a0, "AUC with Stage 1": a1, "gain": a1 - a0})
E6 = pd.DataFrame(rows)
sm = E6.groupby(["model", "system", "Stage-2 training size"]).agg(without=("AUC without Stage 1", "mean"), with_s1=("AUC with Stage 1", "mean"), gain=("gain", "mean"), gain_sd=("gain", "std")).reset_index()
display(sm.pivot_table(index=["model", "system"], columns="Stage-2 training size", values="gain"))
print("mean AUC with / without Stage 1 at the smallest and largest training size:")
display(sm[sm["Stage-2 training size"].isin([1000, len(S2)])])

parts: S1 147,165 | S2 pool 110,380 | T 110,366 rows. Stage-1 R2 on T: 0.506


Stage-2 training size               1000    3000    10000   30000   110380
model               system                                                
Logistic Regression brakes          0.0032  0.0003  0.0001  0.0003  0.0002
                    electrical      0.0006  0.0005  0.0003  0.0003  0.0002
                    engine-related  0.0069  0.0013  0.0003 -0.0003 -0.0003
XGBoost             brakes          0.0040 -0.0029 -0.0019 -0.0019 -0.0001
                    electrical     -0.0007 -0.0005 -0.0008 -0.0004 -0.0000
                    engine-related  0.0000  0.0030 -0.0012  0.0010  0.0009

mean AUC with / without Stage 1 at the smallest and largest training size:


,model,system,Stage-2 training size,without,with_s1,gain,gain_sd
0,Logistic Regression,brakes,1000,0.6300,0.6332,0.0032,0.0012
4,Logistic Regression,brakes,110380,0.6819,0.6821,0.0002,NaN
5,Logistic Regression,electrical,1000,0.6859,0.6865,0.0006,0.0007
9,Logistic Regression,electrical,110380,0.7129,0.7131,0.0002,NaN
10,Logistic Regression,engine-related,1000,0.6533,0.6602,0.0069,0.0047
14,Logistic Regression,engine-related,110380,0.7146,0.7143,-0.0003,NaN
15,XGBoost,brakes,1000,0.5668,0.5709,0.0040,0.0027
19,XGBoost,brakes,110380,0.6831,0.6830,-0.0001,NaN
20,XGBoost,electrical,1000,0.6205,0.6198,-0.0007,0.0010
24,XGBoost,electrical,110380,0.7234,0.7233,-0.0000,NaN


## 7. The same idea across countries: Stage 1 learned from UK odometers, Stage 2 trained on Dutch inspections (the Dutch open data have **no odometer**)
This is the closest available analogue of Israel: Stage 2 never sees an odometer; the only mileage information is what a Stage 1 trained on other data can supply from the registry fields.
Features restricted to the vocabulary common to both countries (age 4-20, petrol/diesel/hybrid, common makes).

In [8]:
AGE, FUELS = (4, 20), ["petrol", "diesel", "hybrid"]
u = uk_all[uk_all.age_year.between(*AGE) & uk_all.fuel.isin(FUELS)].copy(); n_ = nl_all[nl_all.age_year.between(*AGE) & nl_all.fuel.isin(FUELS)].copy()
mu, mn = u.make.value_counts(), n_.make.value_counts(); CM = sorted(set(mu[mu >= 300].index) & set(mn[mn >= 300].index))
for df in (u, n_): df["make_c"] = df.make.where(df.make.isin(CM), "OTHER")
CAT2 = {"fuel": FUELS, "make_c": CM + ["OTHER"]}
def prep2(df, cols):
    X = df[cols].copy()
    for c in cols:
        if c in CAT2: X[c] = pd.Categorical(X[c], categories=CAT2[c])
    return X
F2 = ["age_year", "fuel", "make_c"]
s1uk = s1_model().fit(prep2(u, F2), u.km)                           # Stage 1: all UK vehicles (real odometers)
n_["km_est"] = s1uk.predict(prep2(n_, F2))
print("Stage 1 (UK odometers) applied to Dutch cars: estimated km, median", round(float(n_.km_est.median())), "| range of estimate by age 4/10/20:", n_.groupby("age_year").km_est.mean().loc[[4, 10, 20]].round(0).to_dict())
a_i, t_i = next(GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=0).split(n_, groups=n_.unit)); NA, NT = n_.iloc[a_i], n_.iloc[t_i]
def run2(tr, te, tcol, with_e, kind):
    y = te[tcol].to_numpy()
    if kind == "XGBoost":
        cols = F2 + (["km_est"] if with_e else []); m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, min_child_weight=20, tree_method="hist", enable_categorical=True, n_jobs=8, random_state=0).fit(prep2(tr, cols), tr[tcol])
        return roc_auc_score(y, m.predict_proba(prep2(te, cols))[:, 1])
    cols = F2 + (["km_est"] if with_e else []); m = lr_pipe(with_e).fit(tr[cols].astype({"fuel": str, "make_c": str}), tr[tcol])
    return roc_auc_score(y, m.predict_proba(te[cols].astype({"fuel": str, "make_c": str}))[:, 1])
rows = []
for n in [1000, 3000, 10000, 30000, 100000, len(NA)]:
    for rep in range(3 if n < len(NA) else 1):
        tr = NA.sample(n, random_state=rep) if n < len(NA) else NA
        for tname, tcol in TG.items():
            for kind in ["XGBoost", "Logistic Regression"]:
                a0, a1 = run2(tr, NT, tcol, False, kind), run2(tr, NT, tcol, True, kind)
                rows.append({"Stage-2 training size": n, "rep": rep, "system": tname, "model": kind, "AUC without Stage 1": a0, "AUC with Stage 1": a1, "gain": a1 - a0})
E7 = pd.DataFrame(rows)
display(E7.pivot_table(index=["model", "system"], columns="Stage-2 training size", values="gain"))
print("AUC with / without Stage 1 (UK-trained) on Dutch test cars, smallest and largest training set:")
display(E7.groupby(["model", "system", "Stage-2 training size"]).agg(without=("AUC without Stage 1", "mean"), with_s1=("AUC with Stage 1", "mean"), gain=("gain", "mean")).reset_index().query("`Stage-2 training size` in [1000, %d]" % len(NA)))

Stage 1 (UK odometers) applied to Dutch cars: estimated km, median 102199 | range of estimate by age 4/10/20: {4: 49557.0, 10: 110172.0, 20: 151122.0}


Stage-2 training size               1000    3000    10000   30000   100000  450205
model               system                                                        
Logistic Regression brakes          0.0037 -0.0003 -0.0002  0.0001  0.0001 -0.0002
                    electrical     -0.0003  0.0004  0.0000 -0.0001  0.0001  0.0000
                    engine-related  0.0072  0.0013 -0.0012 -0.0006 -0.0002 -0.0002
XGBoost             brakes          0.0006 -0.0068 -0.0048 -0.0023 -0.0013  0.0000
                    electrical     -0.0047 -0.0078 -0.0048 -0.0023 -0.0005  0.0003
                    engine-related  0.0000  0.0023 -0.0005 -0.0030 -0.0007  0.0009

AUC with / without Stage 1 (UK-trained) on Dutch test cars, smallest and largest training set:


,model,system,Stage-2 training size,without,with_s1,gain
0,Logistic Regression,brakes,1000,0.6145,0.6182,0.0037
5,Logistic Regression,brakes,450205,0.6610,0.6608,-0.0002
6,Logistic Regression,electrical,1000,0.6433,0.6431,-0.0003
11,Logistic Regression,electrical,450205,0.6748,0.6748,0.0000
12,Logistic Regression,engine-related,1000,0.7029,0.7102,0.0072
17,Logistic Regression,engine-related,450205,0.7657,0.7655,-0.0002
18,XGBoost,brakes,1000,0.5899,0.5905,0.0006
23,XGBoost,brakes,450205,0.6654,0.6654,0.0000
24,XGBoost,electrical,1000,0.6250,0.6203,-0.0047
29,XGBoost,electrical,450205,0.6765,0.6768,0.0003
